<a href="https://colab.research.google.com/github/lukmannm/data-science-2026/blob/main/Pertemuan3_Lukman_240401010181.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **PERTEMUAN 3: Data Cleaning: Missing Values, Outlier & Ekstraksi Data**

- Nama: Lukman Nur Hakim
- NIM: 240401010181
- Kelas: IF401
- Mata Kuliah: Data Science


## Import Library

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats.mstats import winsorize
import requests

print("Library berhasil diimport!")

Library berhasil diimport!


## Load Dataset

In [ ]:
df = pd.read_csv('/content/housing_dirty.csv')
print(f'Dataset berhasil dimuat: {df.shape[0]} baris, {df.shape[1]} kolom')
df.head(10)

Dataset berhasil dimuat: 130 baris, 7 kolom


,id,luas_m2,harga_juta,kota,kamar,tahun_bangun,kondisi
0,1,297.0,1084.0,jogja,2.0,2000,baik
1,2,254.0,761.0,Medan,NaN,1995,Bagus
2,3,249.7,895.0,Depok,NaN,1983,baik
3,4,49.7,178.0,YGY,5.0,2013,baik
4,5,133.4,424.0,Medan,5.0,2004,Sedang
5,6,153.3,814.0,Jakarta,1.0,2006,Sedang
6,7,114.3,NaN,jakarta,3.0,2011,baik
7,8,NaN,333.0,Yogyakarta,NaN,1989,baik sekali
8,9,81.2,307.0,Yogyakarta,1.0,1996,SEDANG
9,10,69.1,237.0,Bandung,5.0,1980,baik


## Eksplorasi Awal

In [ ]:
# Info umum dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            130 non-null    int64  
 1   luas_m2       112 non-null    float64
 2   harga_juta    113 non-null    float64
 3   kota          130 non-null    object 
 4   kamar         120 non-null    float64
 5   tahun_bangun  130 non-null    int64  
 6   kondisi       130 non-null    object 
dtypes: float64(3), int64(2), object(2)
memory usage: 7.2+ KB


### Missing Values per Kolom

In [ ]:
print(df.isnull().sum())

id               0
luas_m2         18
harga_juta      17
kota             0
kamar           10
tahun_bangun     0
kondisi          0
dtype: int64


### Cek Duplikat

In [ ]:
print(f'Jumlah baris duplikat: {df.duplicated().sum()}')
print(f'Jumlah total baris: {len(df)}')

Jumlah baris duplikat: 0
Jumlah total baris: 130


## Nilai Unik Kolom String (Cek Inkonsistensi)

In [ ]:
print('Nilai unik kota:')
for kota in df['kota'].unique():
    print(f'  - {kota}')

print('\nNilai unik kondisi:')
for kondisi in df['kondisi'].unique():
    print(f'  - {kondisi}')

Nilai unik kota:
  - jogja
  - Medan
  - Depok
  - YGY
  - Jakarta
  - jakarta
  - Yogyakarta
  - Bandung
  - Surabaya
  - dpk
  - sby
  - Makassar
  - mdn
  - medan
  - Semarang
  - semarang
  - yogyakarta
  - Jogja
  - JAKARTA
  - Smg
  - DEPOK
  - Bdg
  - makassar
  - surabaya
  - MAKASSAR
  - depok
  - bandung
  - Bandung 
  - SURABAYA
  - Mksr
  -  Jakarta

Nilai unik kondisi:
  - baik
  - Bagus
  - Sedang
  - baik sekali
  - SEDANG
  - sedang
  - BAIK
  - rusak
  - cukup
  - Baik
  - Cukup
  - perlu renovasi
  - bagus
  - jelek
  - RUSAK


## Hapus Baris Duplikat

In [ ]:
sebelum = len(df)
df.drop_duplicates(inplace=True)
sesudah = len(df)

print(f'Baris sebelum : {sebelum}')
print(f'Baris sesudah : {sesudah}')
print(f'Duplikat dihapus : {sebelum - sesudah} baris')

Baris sebelum : 130
Baris sesudah : 130
Duplikat dihapus : 0 baris


## Normalisasi String

In [ ]:
# kota: title case (huruf pertama tiap kata kapital)
df['kota'] = df['kota'].str.strip().str.title()

# kondisi: lowercase semua
df['kondisi'] = df['kondisi'].str.strip().str.lower()

# Cek hasil
print('Nilai unik kota setelah normalisasi:')
for kota in df['kota'].unique():
    print(f'  - {kota}')

print('\nNilai unik kondisi setelah normalisasi:')
for kondisi in df['kondisi'].unique():
    print(f'  - {kondisi}')

Nilai unik kota setelah normalisasi:
  - Jogja
  - Medan
  - Depok
  - Ygy
  - Jakarta
  - Yogyakarta
  - Bandung
  - Surabaya
  - Dpk
  - Sby
  - Makassar
  - Mdn
  - Semarang
  - Smg
  - Bdg
  - Mksr

Nilai unik kondisi setelah normalisasi:
  - baik
  - bagus
  - sedang
  - baik sekali
  - rusak
  - cukup
  - perlu renovasi
  - jelek


## Imputasi Missing Values

### Sebelum Imputasi

In [ ]:
print('Missing values sebelum imputasi:')
print(df.isnull().sum())

Missing values sebelum imputasi:
id               0
luas_m2         18
harga_juta      17
kota             0
kamar           10
tahun_bangun     0
kondisi          0
dtype: int64


### Proses Imputasi

In [ ]:
# Kolom numerik diisi dengan median
num_cols = ['luas_m2', 'harga_juta', 'kamar']
for col in num_cols:
    median_val = df[col].median()
    missing_count = df[col].isnull().sum()
    df[col] = df[col].fillna(median_val)
    print(f'[{col}] median={median_val:.2f} | {missing_count} nilai diisi')

print()

# Kolom kategorik diisi dengan modus
cat_cols = ['kota', 'kondisi']
for col in cat_cols:
    modus_val = df[col].mode()[0]
    missing_count = df[col].isnull().sum()
    df[col] = df[col].fillna(modus_val)
    print(f'[{col}] modus="{modus_val}" | {missing_count} nilai diisi')

[luas_m2] median=193.80 | 18 nilai diisi
[harga_juta] median=655.00 | 17 nilai diisi
[kamar] median=4.00 | 10 nilai diisi

[kota] modus="Bandung" | 0 nilai diisi
[kondisi] modus="baik" | 0 nilai diisi


### Setelah Imputasi

In [ ]:
print('Missing values setelah imputasi:')
print(df.isnull().sum())

Missing values setelah imputasi:
id              0
luas_m2         0
harga_juta      0
kota            0
kamar           0
tahun_bangun    0
kondisi         0
dtype: int64


## Tangani Outlier dengan IQR Fence

### Sebelum Clip

In [ ]:
print('Statistik sebelum clip:')
print(df[['harga_juta', 'luas_m2', 'tahun_bangun']].describe())


Statistik sebelum clip:
         harga_juta      luas_m2  tahun_bangun
count  1.300000e+02   130.000000    130.000000
mean   7.699047e+05   257.405385   2062.638462
std    8.770521e+06   821.951924    701.684043
min   -5.000000e+02   -50.000000   1890.000000
25%    3.805000e+02   101.600000   1991.250000
50%    6.550000e+02   193.800000   2002.000000
75%    9.160000e+02   266.150000   2011.750000
max    1.000000e+08  9500.000000   9999.000000


### Proses IQR Fence

In [ ]:
def iqr_fence_clip(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outlier_count = ((df[col] < lower) | (df[col] > upper)).sum()
    df[col] = df[col].clip(lower=lower, upper=upper)

    print(f'[{col}]')
    print(f'  Q1={Q1:.2f}, Q3={Q3:.2f}, IQR={IQR:.2f}')
    print(f'  Fence: [{lower:.2f}, {upper:.2f}]')
    print(f'  Outlier di-clip: {outlier_count}')

iqr_fence_clip(df, 'harga_juta')
iqr_fence_clip(df, 'luas_m2')

outlier_tahun = (df['tahun_bangun'] < 1950).sum()
df['tahun_bangun'] = df['tahun_bangun'].clip(lower=1950)
print(f'\n[tahun_bangun]')
print(f'  Nilai < 1950 di-clip ke 1950')
print(f'  Outlier di-clip: {outlier_tahun}')

[harga_juta]
  Q1=380.50, Q3=916.00, IQR=535.50
  Fence: [-422.75, 1719.25]
  Outlier di-clip: 3
[luas_m2]
  Q1=101.60, Q3=266.15, IQR=164.55
  Fence: [-145.22, 512.97]
  Outlier di-clip: 1

[tahun_bangun]
  Nilai < 1950 di-clip ke 1950
  Outlier di-clip: 1


## Validasi Akhir

In [ ]:
total_missing = df.isnull().sum().sum()
total_duplikat = df.duplicated().sum()

print(f'Total missing values : {total_missing}  (target: 0)')
print(f'Total baris duplikat : {total_duplikat}  (target: 0)')
print(f'Total baris akhir    : {len(df)}')

if total_missing == 0 and total_duplikat == 0:
    print('\n Dataset bersih dan siap digunakan!')
else:
    print('\n Masih ada masalah yang belum ditangani.')

Total missing values : 0  (target: 0)
Total baris duplikat : 0  (target: 0)
Total baris akhir    : 130

 Dataset bersih dan siap digunakan!


## Export Dataset Bersih

In [ ]:
df.to_csv('housing_clean.csv', index=False)
print('Dataset berhasil disimpan sebagai housing_clean.csv')
df.head()

Dataset berhasil disimpan sebagai housing_clean.csv


,id,luas_m2,harga_juta,kota,kamar,tahun_bangun,kondisi
0,1,297.0,1084.0,jogja,2.0,2000,baik
1,2,254.0,761.0,Medan,4.0,1995,Bagus
2,3,249.7,895.0,Depok,4.0,1983,baik
3,4,49.7,178.0,YGY,5.0,2013,baik
4,5,133.4,424.0,Medan,5.0,2004,Sedang


## Akses API JSONPlaceholder

In [ ]:
try:
    params = {'userId': 1}  # filter by user
    response = requests.get('https://jsonplaceholder.typicode.com/posts', params=params)

    if response.status_code == 200:
        print(f'Request berhasil! Status code: {response.status_code}')

        df_posts = pd.DataFrame(response.json())
        display(df_posts)
    else:
        print(f'Request gagal! Status code: {response.status_code}')

except Exception as e:
    print(f'Terjadi error: {e}')

Request berhasil! Status code: 200


,userId,id,title,body
0,1,1,sunt aut facere repellat provident occaecati e...,quia et suscipit\nsuscipit recusandae consequu...
1,1,2,qui est esse,est rerum tempore vitae\nsequi sint nihil repr...
2,1,3,ea molestias quasi exercitationem repellat qui...,et iusto sed quo iure\nvoluptatem occaecati om...
3,1,4,eum et est occaecati,ullam et saepe reiciendis voluptatem adipisci\...
4,1,5,nesciunt quas odio,repudiandae veniam quaerat sunt sed\nalias aut...
5,1,6,dolorem eum magni eos aperiam quia,ut aspernatur corporis harum nihil quis provid...
6,1,7,magnam facilis autem,dolore placeat quibusdam ea quo vitae\nmagni q...
7,1,8,dolorem dolore est ipsam,dignissimos aperiam dolorem qui eum\nfacilis q...
8,1,9,nesciunt iure omnis dolorem tempora et accusan...,consectetur animi nesciunt iure dolore\nenim q...
9,1,10,optio molestias id quia eum,quo et expedita modi cum officia vel magni\ndo...


## Kesimpulan

Praktikum ini mempelajari proses data cleaning secara lengkap, mulai dari menghapus duplikat, normalisasi string, imputasi missing values dengan median dan modus, hingga penanganan outlier menggunakan metode IQR fence. Dataset housing_dirty.csv berhasil dibersihkan tanpa missing values dan duplikat, serta outlier pada kolom harga dan luas sudah di-clip ke batas wajar. Keterbatasan yang muncul adalah metode imputasi median bersifat sederhana dan belum mempertimbangkan relasi antar fitur, sehingga nilai yang diisi belum tentu mencerminkan kondisi properti yang sebenarnya.